In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

from src.config import ALL_ASSETS, INITIAL_CAPITAL, RISK_PROFILES
from src.data_loader import get_prices, get_returns
from src.backtester import run_backtest
from src.optimization.portfolio_optimizer import optimize_for_profile
from src.risk_manager import (
    portfolio_var_cvar, run_stress_tests, risk_budget_allocation
)

prices  = get_prices(ALL_ASSETS)
returns = get_returns(prices)

# Correr backtest dos 3 perfis
profile_results = {}
for p in ["low", "medium", "high"]:
    profile_results[p] = run_backtest(
        prices,
        weight_fn=lambda px, pr=p: optimize_for_profile(px, pr),
        initial_capital=INITIAL_CAPITAL,
        rebalance_freq="monthly",
    )

[cache] 2190 dias x 8 ativos


#### Célula 2 — VaR e CVaR dos 3 perfis:

In [2]:
rows = []
for profile in ["low", "medium", "high"]:
    w   = optimize_for_profile(prices, profile)
    var = portfolio_var_cvar(w, returns, capital=INITIAL_CAPITAL)
    rows.append({
        "Perfil":           profile.upper(),
        "VaR diário %":     var["daily_var_%"],
        "VaR diário EUR":   var["daily_var_eur"],
        "CVaR diário %":    var["daily_cvar_%"],
        "CVaR diário EUR":  var["daily_cvar_eur"],
        "VaR anual %":      var["annual_var_%"],
    })

pd.DataFrame(rows).set_index("Perfil")

,VaR diário %,VaR diário EUR,CVaR diário %,CVaR diário EUR,VaR anual %
Perfil,,,,,
LOW,0.937,93.70,1.498,149.78,14.87
MEDIUM,1.397,139.68,2.289,228.93,22.17
HIGH,0.949,94.93,1.517,151.68,15.07


#### Célula 3 — Stress tests comparados:

In [3]:
fig = go.Figure()
profiles = ["low", "medium", "high"]
colors   = {"low": "green", "medium": "blue", "high": "orange"}

for profile in profiles:
    w      = optimize_for_profile(prices, profile)
    stress = run_stress_tests(w, capital=INITIAL_CAPITAL)
    fig.add_trace(go.Bar(
        name=profile.upper(),
        x=stress["cenario"],
        y=stress["perda_%"],
        marker_color=colors[profile],
        text=stress["perda_%"].apply(lambda x: f"{x:.1f}%"),
        textposition="outside",
    ))

fig.update_layout(
    title="Stress tests — perda estimada (%) por perfil e cenário",
    barmode="group",
    yaxis_title="Perda (%)",
    xaxis_tickangle=-20,
    height=500,
)
fig.show()

#### Célula 4 — Alocação dos €10.000:

In [4]:
# Define quanto queres em cada perfil
investor_split = {"low": 0.40, "medium": 0.40, "high": 0.20}

# Métricas de backtest de cada perfil
profile_metrics = {p: profile_results[p]["metrics"] for p in ["low","medium","high"]}

budget = risk_budget_allocation(INITIAL_CAPITAL, investor_split, profile_metrics)
print(budget.to_string(index=False))

# Capital por ativo
print("\nCapital por ativo:")
for profile, pct in investor_split.items():
    cap_p = INITIAL_CAPITAL * pct
    w     = optimize_for_profile(prices, profile)
    print(f"\n  {profile.upper()} (EUR {cap_p:,.0f}):")
    for ticker, weight in sorted(w.items(), key=lambda x: -x[1]):
        if weight > 0.005:
            print(f"    {ticker:10s}  {weight*100:5.1f}%  "
                  f"EUR {cap_p*weight:,.0f}")

perfil  alocacao_%  capital_eur  cagr_esperado_%  vol_anual_%  max_drawdown_%  sharpe
   LOW        40.0       4000.0            70.14        43.63          -15.26   1.562
MEDIUM        40.0       4000.0            81.11        44.76          -13.32   1.767
  HIGH        20.0       2000.0            71.50        44.29          -22.06   1.569

Capital por ativo:

  LOW (EUR 4,000):
    TLT          39.5%  EUR 1,579
    SPY          37.7%  EUR 1,509
    USO          11.9%  EUR 475
    GLD           9.8%  EUR 391
    AAPL          1.2%  EUR 46

  MEDIUM (EUR 4,000):
    AAPL         35.0%  EUR 1,400
    SPY          29.4%  EUR 1,176
    GLD          22.8%  EUR 911
    BTC-USD       7.0%  EUR 282
    TLT           5.8%  EUR 232

  HIGH (EUR 2,000):
    TLT          40.6%  EUR 812
    SPY          17.1%  EUR 342
    GLD          14.1%  EUR 282
    USO          11.9%  EUR 238
    QQQ           9.4%  EUR 188
    AAPL          4.7%  EUR 93
    BTC-USD       1.6%  EUR 33
    ETH-USD       0.6% 